# Lab 1 — One detection, three clouds (≈10 min)

The same attack hit the `payments-prod` Kubernetes cluster on **AWS EKS**, **GCP GKE** and **Azure AKS**.
Every cloud logged it. Every cloud logged it **differently**.

You will write two detections against each cloud's raw audit logs:

| Rule | Question |
|---|---|
| **A. Privileged pod** | Did anyone create a pod with `securityContext.privileged: true`? |
| **B. Pod exec** | Did anyone `kubectl exec` into a pod? |

Run each cell with **Shift+Enter**. Edit the `-- TODO` queries until `lab1.check(...)` prints ✅.

> ⚠️ **DuckDB gotcha:** `->>` binds *looser* than `AND` and `=`, so
> `event->>'$.verb' = 'create' AND event->>'$.x' = 'y'` silently parses wrong.
> Use `json_extract_string(event, '$.path')` (as the starters do) or wrap every `->>` in parentheses.
> JSONPath arrays start at **0** (`$.sourceIPs[0]`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
from workshop import lab1
con = lab1.setup()

## Step 1 — Look before you query

One raw record per cloud. Note **where the Kubernetes audit event lives** in each, and what the envelope adds.

- **EKS**: CloudWatch Logs → the native audit `Event`, plus `aws_account` and `log_group`
- **AKS**: Azure diagnostics → the native audit `Event` from `properties.log`, plus an upper-cased `resource_id`
- **GKE**: Cloud Logging → no native `Event`; Google re-shapes it into `protoPayload`

In [ ]:
lab1.sample("eks")

In [ ]:
lab1.sample("aks")

In [ ]:
lab1.sample("gke")

## Rule A — Privileged pod created

A privileged container runs with full access to the node. Combined with `hostPID` and a host-path mount,
it is a container escape waiting to happen (MITRE **T1611**).

**EKS is done for you** — read it, run it, then adapt it for the other two clouds.

In [ ]:
PRIVILEGED_POD_EKS = """
SELECT json_extract_string(event, '$.auditID') AS event_id,
       json_extract_string(event, '$.user.username') AS principal,
       json_extract_string(event, '$.objectRef.namespace') AS namespace,
       json_extract_string(event, '$.requestObject.spec.containers[0].image') AS image
FROM raw_eks
WHERE json_extract_string(event, '$.verb') = 'create'
  AND json_extract_string(event, '$.objectRef.resource') = 'pods'
  AND json_extract_string(event, '$.objectRef.subresource') IS NULL
  AND json_extract_string(event, '$.requestObject.spec.containers[0].securityContext.privileged') = 'true'
"""
lab1.check("privileged_pod", "eks", PRIVILEGED_POD_EKS)
lab1.q(PRIVILEGED_POD_EKS)

### AKS
The audit event inside `properties.log` is the same native Kubernetes Event as EKS - start from the EKS query. What actually changes?

In [ ]:
PRIVILEGED_POD_AKS = """
SELECT json_extract_string(event, '$.auditID') AS event_id,
       json_extract_string(event, '$.user.username') AS principal,
       json_extract_string(event, '$.objectRef.namespace') AS namespace,
       json_extract_string(event, '$.requestObject.spec.containers[0].image') AS image
FROM raw_aks
WHERE json_extract_string(event, '$.verb') = 'create'
  AND json_extract_string(event, '$.objectRef.resource') = 'pods'
  AND json_extract_string(event, '$.objectRef.subresource') IS NULL
  AND json_extract_string(event, '$.requestObject.spec.containers[0].securityContext.privileged') = 'true'
"""
lab1.check("privileged_pod", "aks", PRIVILEGED_POD_AKS)
lab1.q(PRIVILEGED_POD_AKS)

### GKE
GKE has no `verb` or `objectRef`. Look at `protoPayload.methodName` and `protoPayload.resourceName` in the sample above. The pod spec is under `protoPayload.request`.

In [ ]:
PRIVILEGED_POD_GKE = """
SELECT json_extract_string(entry, '$.operation.id') AS event_id,
       json_extract_string(entry, '$.protoPayload.authenticationInfo.principalEmail') AS principal,
       split_part(json_extract_string(entry, '$.protoPayload.resourceName'), '/', 4) AS namespace,
       json_extract_string(entry, '$.protoPayload.request.spec.containers[0].image') AS image
FROM raw_gke
WHERE json_extract_string(entry, '$.protoPayload.methodName') = 'io.k8s.core.v1.pods.create'
  AND json_extract_string(entry, '$.protoPayload.request.spec.containers[0].securityContext.privileged') = 'true'
"""
lab1.check("privileged_pod", "gke", PRIVILEGED_POD_GKE)
lab1.q(PRIVILEGED_POD_GKE)

Hits: **EKS 162, GKE 161, AKS 161**, and only **1 per cloud** is the attack.
The rest are the cloud provider's own networking/storage DaemonSets (privileged by design, recreated every time
a node joins) and, on EKS, an SRE's `kubectl debug node` pod. Raw logs can't tell them apart.
Hold that thought until Lab 4.

## Rule B — Exec into a pod

`kubectl exec` gives an interactive shell inside a running container (MITRE **T1609**).
This time you write all three. Expect **EKS 13, GKE 11, AKS 12** hits.

### EKS
`kubectl exec` is a `create` on the `exec` **subresource** of a pod.

In [ ]:
POD_EXEC_EKS = """
SELECT json_extract_string(event, '$.auditID') AS event_id,
       json_extract_string(event, '$.user.username') AS principal,
       json_extract_string(event, '$.sourceIPs[0]') AS src_ip,
       json_extract_string(event, '$.objectRef.name') AS pod
FROM raw_eks
WHERE json_extract_string(event, '$.verb') = 'create'
  AND json_extract_string(event, '$.objectRef.resource') = 'pods'
  AND json_extract_string(event, '$.objectRef.subresource') = 'exec'
"""
lab1.check("pod_exec", "eks", POD_EXEC_EKS)
lab1.q(POD_EXEC_EKS)

### AKS
Same shape as EKS.

In [ ]:
POD_EXEC_AKS = """
SELECT json_extract_string(event, '$.auditID') AS event_id,
       json_extract_string(event, '$.user.username') AS principal,
       json_extract_string(event, '$.sourceIPs[0]') AS src_ip,
       json_extract_string(event, '$.objectRef.name') AS pod
FROM raw_aks
WHERE json_extract_string(event, '$.verb') = 'create'
  AND json_extract_string(event, '$.objectRef.resource') = 'pods'
  AND json_extract_string(event, '$.objectRef.subresource') = 'exec'
"""
lab1.check("pod_exec", "aks", POD_EXEC_AKS)
lab1.q(POD_EXEC_AKS)

### GKE
On GKE the subresource is folded into `methodName`. Run `lab1.q("SELECT DISTINCT json_extract_string(entry, '$.protoPayload.methodName') m FROM raw_gke ORDER BY m", 100)` to see every method name.

In [ ]:
POD_EXEC_GKE = """
SELECT json_extract_string(entry, '$.operation.id') AS event_id,
       json_extract_string(entry, '$.protoPayload.authenticationInfo.principalEmail') AS principal,
       json_extract_string(entry, '$.protoPayload.requestMetadata.callerIp') AS src_ip,
       split_part(json_extract_string(entry, '$.protoPayload.resourceName'), '/', 6) AS pod
FROM raw_gke
WHERE json_extract_string(entry, '$.protoPayload.methodName') = 'io.k8s.core.v1.pods.exec.create'
"""
lab1.check("pod_exec", "gke", POD_EXEC_GKE)
lab1.q(POD_EXEC_GKE)

## Stretch — Only the execs that escape to the host

Most execs are engineers opening `sh` or `bash` to debug. One per cloud is the attacker running
`nsenter --target 1` to jump into the node's namespaces. Narrow Rule B to the host-escape only.

### EKS
The exec command is only recorded as `command=` query parameters on `requestURI`. Look for `nsenter`, which enters the host's namespaces.

In [ ]:
HOST_ESCAPE_EXEC_EKS = """
SELECT json_extract_string(event, '$.auditID') AS event_id,
       json_extract_string(event, '$.user.username') AS principal,
       json_extract_string(event, '$.sourceIPs[0]') AS src_ip,
       json_extract_string(event, '$.objectRef.name') AS pod
FROM raw_eks
WHERE json_extract_string(event, '$.verb') = 'create'
  AND json_extract_string(event, '$.objectRef.resource') = 'pods'
  AND json_extract_string(event, '$.objectRef.subresource') = 'exec'
  AND json_extract_string(event, '$.requestURI') LIKE '%command=nsenter%'
"""
lab1.check("host_escape_exec", "eks", HOST_ESCAPE_EXEC_EKS)
lab1.q(HOST_ESCAPE_EXEC_EKS)

### AKS
Same as EKS.

In [ ]:
HOST_ESCAPE_EXEC_AKS = """
SELECT json_extract_string(event, '$.auditID') AS event_id,
       json_extract_string(event, '$.user.username') AS principal,
       json_extract_string(event, '$.sourceIPs[0]') AS src_ip,
       json_extract_string(event, '$.objectRef.name') AS pod
FROM raw_aks
WHERE json_extract_string(event, '$.verb') = 'create'
  AND json_extract_string(event, '$.objectRef.resource') = 'pods'
  AND json_extract_string(event, '$.objectRef.subresource') = 'exec'
  AND json_extract_string(event, '$.requestURI') LIKE '%command=nsenter%'
"""
lab1.check("host_escape_exec", "aks", HOST_ESCAPE_EXEC_AKS)
lab1.q(HOST_ESCAPE_EXEC_AKS)

### GKE — try it

Find the attacker's exec on GKE and look for the command. Where did it go?

<details><summary>Answer</summary>

GKE's audit log never records the request URI, so the `nsenter` arguments **do not exist** in GKE logs.
No query can recover them. Some detections are not portable *at all*, no matter how you normalize:
know what each source can and cannot see.
</details>

In [ ]:
lab1.q("""
SELECT json_extract_string(entry, '$.protoPayload.methodName') AS method,
       json_extract_string(entry, '$.protoPayload.resourceName') AS resource,
       json_extract(entry, '$.protoPayload') AS payload
FROM raw_gke
WHERE json_extract_string(entry, '$.protoPayload.methodName') LIKE '%exec%'
""")

## Wrap-up

In [ ]:
lab1.scorecard()

**Count what you just maintained:** 2 rules × 3 clouds = **6 queries**, built on 3 different field
layouts, plus one rule that works on only 2 of the 3 clouds. Add a fourth cloud, or change the definition of
"privileged" (say, to include `hostPID` without `privileged`), and you edit every one of them.

| | EKS | AKS | GKE |
|---|---|---|---|
| Where the event lives | `logEvents[].message` (JSON string) | `properties.log` (JSON string) | `protoPayload` (re-shaped) |
| Verb | `verb` | `verb` | last segment of `methodName` |
| Resource / subresource | `objectRef.resource` / `.subresource` | same | packed into `methodName` and `resourceName` |
| Who | `user.username` (an IAM role ARN) | `user.username` (UPN or GUID) | `authenticationInfo.principalEmail` |
| Request URI | ✅ | ✅ | ❌ never logged |

**Next — Lab 2:** map all three into one schema (OCSF), so Lab 3 can write each rule **once**.